# 11-3. 종일 Poisson QMLE 모형 튜닝

2025년 종일 서울행 광역버스 승차인원을 종속변수로 하고 `log(2024년 승차인원)`을 offset으로 적용한다. 분석 단위는 셀×노선이며 노선 고정효과와 셀 단위 군집 강건 표준오차를 사용한다. 모든 연속형 변수는 표준화하지 않은 원단위이다.

Poisson의 평균·분산 동일 가정이 충족되지 않더라도 조건부 평균식이 올바르면 QMLE 계수는 일관될 수 있다. 추론에는 셀 군집 강건 표준오차를 사용한다.


In [ ]:
import sys,importlib
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
PROJECT_DIR=Path.cwd().parent if Path.cwd().name=='analysis' else Path.cwd()
sys.path.insert(0,str(PROJECT_DIR))
import config as cfg
cfg=importlib.reload(cfg); cfg.setup_notebook()
ID,ROUTE='셀 ID','노선'
print('설정 파일:',Path(cfg.__file__).resolve())


## 1. 공통 데이터 구축

In [ ]:
grid=cfg.load_model_data(); routes=cfg.load_cell_route_all_day()
grid_cols=[ID,'walk_20','walk_20_60','ic_access','pop_20_50','buildings']
model_data=routes.merge(grid[grid_cols],on=ID,how='inner',validate='many_to_one')
pop=cfg.read_csv(cfg.POPULATION_CHANGE_PATH); pop[ID]=pd.to_numeric(pop[ID],errors='coerce').astype('Int64')
pop_col=next(c for c in pop.columns if '증감(100명)' in str(c)); pop['pop_change']=pd.to_numeric(pop[pop_col],errors='coerce')*100
model_data=model_data.merge(pop[[ID,'pop_change']],on=ID,how='left',validate='many_to_one')
numeric_cols=['y2024','y2025','walk_20','walk_20_60','ic_access','competing_share','pop_20_50','pop_change','buildings']
for c in numeric_cols: model_data[c]=pd.to_numeric(model_data[c],errors='coerce')
model_data=model_data[model_data['y2024']>0].dropna(subset=numeric_cols).reset_index(drop=True)
print('관측치:',len(model_data),'| 격자:',model_data[ID].nunique(),'| 노선:',model_data[ROUTE].nunique())


## 2. Poisson QMLE 추정 함수

In [ ]:
LABELS={'walk_20':'GTX 보행 20분 이내','walk_20_60':'GTX 보행 20~60분','ic_access':'IC까지 도로거리(km)','competing_share':'GTX 경쟁 목적지 비율(%p)','pop_20_50':'2024년 20~50대 인구(명)','pop_change':'2024→2025년 20~50대 인구 증감(명)','buildings':'2024년 건축물 수(동)','land_price_2024_million':'2024년 공시지가(백만원/㎡)','log_land_price_2024':'log(2024년 공시지가)'}
def fit_poisson_qmle(name,features):
    d=model_data.dropna(subset=['y2024','y2025']+features).copy(); x=d[features].astype(float)
    route_fe=pd.get_dummies(d[ROUTE].astype(str),prefix='노선',drop_first=True,dtype=float)
    X=sm.add_constant(pd.concat([x,route_fe],axis=1),has_constant='add'); offset=np.log(d['y2024'].astype(float)); groups=pd.factorize(d[ID])[0]
    fit=sm.GLM(d['y2025'],X,family=sm.families.Poisson(),offset=offset).fit(cov_type='cluster',cov_kwds={'groups':groups,'use_correction':True})
    null=sm.GLM(d['y2025'],np.ones((len(d),1)),family=sm.families.Poisson(),offset=offset).fit(); ci=fit.conf_int().loc[features]
    result=pd.DataFrame({'변수':[LABELS[v] for v in features],'계수':[fit.params[v] for v in features],'IRR':[np.exp(fit.params[v]) for v in features],'상대적 승차인원 차이(%)':[(np.exp(fit.params[v])-1)*100 for v in features],'z값':[fit.tvalues[v] for v in features],'p-value':[fit.pvalues[v] for v in features],'95% CI 하한(%)':[(np.exp(ci.loc[v,0])-1)*100 for v in features],'95% CI 상한(%)':[(np.exp(ci.loc[v,1])-1)*100 for v in features]})
    bic_llf=-2*fit.llf+len(fit.params)*np.log(len(d))
    info={'모형':name,'표본 수':len(d),'격자 수':d[ID].nunique(),'노선 수':d[ROUTE].nunique(),'로그우도':fit.llf,'Deviance 설명비율':1-fit.deviance/null.deviance,'AIC':fit.aic,'BIC(logLik)':bic_llf,'Pearson dispersion':fit.pearson_chi2/fit.df_resid,'Deviance/df':fit.deviance/fit.df_resid}
    return {'fit':fit,'data':d,'result':result,'info':info}


## 3. 기본모형

In [ ]:
BASE_FEATURES=['walk_20','walk_20_60','ic_access','competing_share','pop_20_50','buildings']
base_model=fit_poisson_qmle('기본모형',BASE_FEATURES); display(pd.DataFrame([base_model['info']]).round(4)); display(base_model['result'].round(4))


## 4. 튜닝 1 — 보행 20분 이내 더미만

In [ ]:
TUNE1_FEATURES=['walk_20','ic_access','competing_share','pop_20_50','buildings']
tune1_model=fit_poisson_qmle('튜닝1_20분더미만',TUNE1_FEATURES); display(pd.DataFrame([tune1_model['info']]).round(4)); display(tune1_model['result'].round(4))


## 5. 튜닝 2 — 인구증감으로 교체

In [ ]:
TUNE2_FEATURES=['walk_20','walk_20_60','ic_access','competing_share','pop_change','buildings']
tune2_model=fit_poisson_qmle('튜닝2_인구증감',TUNE2_FEATURES); display(pd.DataFrame([tune2_model['info']]).round(4)); display(tune2_model['result'].round(4))


## 6. 튜닝 3 — 2024년 공시지가 원값 추가

In [ ]:
land=cfg.read_csv(cfg.LAND_PRICE_2024_PATH); land[ID]=pd.to_numeric(land[ID],errors='coerce').astype('Int64')
land['land_price_2024_million']=pd.to_numeric(land['2024년 공시지가(원/㎡)'],errors='coerce')/1_000_000
land['log_land_price_2024']=pd.to_numeric(land['log_2024년 공시지가'],errors='coerce')
model_data=model_data.merge(land[[ID,'land_price_2024_million','log_land_price_2024']],on=ID,how='left',validate='many_to_one')
print('공시지가 매칭 격자:',model_data.loc[model_data['land_price_2024_million'].notna(),ID].nunique(),'/',model_data[ID].nunique())
model_data_all=model_data.copy()
model_data=model_data_all.dropna(subset=['land_price_2024_million','log_land_price_2024']).copy()
land_base_model=fit_poisson_qmle('공시지가비교_동일표본기본',BASE_FEATURES)
display(pd.DataFrame([land_base_model['info']]).round(4))
LAND_RAW_FEATURES=BASE_FEATURES+['land_price_2024_million']
tune3_model=fit_poisson_qmle('튜닝3_공시지가원값',LAND_RAW_FEATURES); display(pd.DataFrame([tune3_model['info']]).round(4)); display(tune3_model['result'].round(4))


## 7. 튜닝 4 — log(2024년 공시지가) 추가

In [ ]:
LAND_LOG_FEATURES=BASE_FEATURES+['log_land_price_2024']
tune4_model=fit_poisson_qmle('튜닝4_공시지가로그',LAND_LOG_FEATURES); display(pd.DataFrame([tune4_model['info']]).round(4)); display(tune4_model['result'].round(4))


## 8. 전체 모형 비교

In [ ]:
models=[base_model,tune1_model,tune2_model,tune3_model,tune4_model]
display(pd.DataFrame([m['info'] for m in models]).round(4))
rows=[]
for m in models:
    t=m['result'][['변수','계수','IRR','상대적 승차인원 차이(%)','p-value']].copy(); t.insert(0,'모형',m['info']['모형']); rows.append(t)
display(pd.concat(rows,ignore_index=True).round(4))


## 9. 튜닝 5 — 2024년 광역버스 통행시간 추가

기본모형에 2024년 종일 서울행 이용자의 셀×노선별 **승객가중 중앙값 통행시간**을 추가합니다. 통행시간은 10분 단위이며 표준화하지 않습니다.

In [ ]:
model_data = model_data_all.copy() if 'model_data_all' in globals() else model_data.copy()
travel = cfg.read_csv(cfg.TRAVEL_TIME_2024_PATH)
travel[ID] = pd.to_numeric(travel[ID], errors='coerce').astype('Int64')
travel[ROUTE] = travel[ROUTE].astype(str).str.removesuffix('.0')
travel['travel_time_10min'] = pd.to_numeric(travel['2024년 중앙값 통행시간(10분 단위)'], errors='coerce')
model_data[ROUTE] = model_data[ROUTE].astype(str).str.removesuffix('.0')
model_data = model_data.merge(travel[[ID, ROUTE, 'travel_time_10min']], on=[ID, ROUTE], how='left', validate='one_to_one')
LABELS['travel_time_10min'] = '2024년 광역버스 중앙값 통행시간(10분)'
TRAVEL_FEATURES = BASE_FEATURES + ['travel_time_10min']
tune5_model = fit_poisson_qmle('튜닝5_2024통행시간', TRAVEL_FEATURES)
display(pd.DataFrame([tune5_model['info']]).round(4))
display(tune5_model['result'].round(4))
print('통행시간 계수는 2024년 통행시간이 10분 증가할 때의 상대적 승차인원 차이를 뜻합니다.')


## 10. 튜닝 6 — 통행시간 1분 단위

튜닝 5와 동일한 통행시간을 1분 단위로 표시합니다. 단위만 바뀌므로 적합도와 p값은 동일합니다.

In [ ]:
model_data['travel_time_min'] = model_data['travel_time_10min'] * 10
LABELS['travel_time_min'] = '2024년 광역버스 중앙값 통행시간(1분)'
TRAVEL_MIN_FEATURES = BASE_FEATURES + ['travel_time_min']
tune6_model = fit_poisson_qmle('튜닝6_2024통행시간_1분단위', TRAVEL_MIN_FEATURES)
display(pd.DataFrame([tune6_model['info']]).round(4))
display(tune6_model['result'].round(4))
print('통행시간 효과는 1분 증가 기준입니다. 10분 효과는 exp(1분 계수 × 10)-1로 계산합니다.')


## 11. 튜닝 7 — 평균 통행시간 비교

기존 통행시간은 승객가중 중앙값입니다. 강건성 비교를 위해 승객가중 평균 통행시간을 10분 단위로 추가합니다.

In [ ]:
travel_mean = cfg.read_csv(cfg.TRAVEL_TIME_2024_PATH)
travel_mean[ID] = pd.to_numeric(travel_mean[ID], errors='coerce').astype('Int64')
travel_mean[ROUTE] = travel_mean[ROUTE].astype(str).str.removesuffix('.0')
travel_mean['travel_mean_10min'] = pd.to_numeric(travel_mean['2024년 승객가중 평균 통행시간(분)'], errors='coerce') / 10
model_data = model_data.drop(columns=['travel_mean_10min'], errors='ignore').merge(travel_mean[[ID, ROUTE, 'travel_mean_10min']], on=[ID, ROUTE], how='left', validate='one_to_one')
LABELS['travel_mean_10min'] = '2024년 광역버스 평균 통행시간(10분)'
TRAVEL_MEAN_FEATURES = BASE_FEATURES + ['travel_mean_10min']
tune7_model = fit_poisson_qmle('튜닝7_2024평균통행시간', TRAVEL_MEAN_FEATURES)
display(pd.DataFrame([tune7_model['info']]).round(4))
display(tune7_model['result'].round(4))
travel_compare = pd.DataFrame([{'통행시간 지표':'승객가중 중앙값', **tune5_model['info']}, {'통행시간 지표':'승객가중 평균', **tune7_model['info']}])
display(travel_compare.round(4))
